# W07 · Python 으로 MD 자동화 (15분)

**무엇을 하나** — 매개변수 표 6줄 → 입력 파일 6개 → LAMMPS 반복 실행(실패 1건은 Δt 절반으로 재시도) → 로그 파싱 → `status.csv` · `summary.json`.

**왜 하나** — W06 에서 손으로 쓴 in 파일을 표에서 **생성**한다. 조성이 수백 개가 되어도 같은 세 스크립트로 돈다 (W16 파이프라인의 ② · ③ 칸).

**실행** — week07 폴더의 Claude Code 에 「`@notebooks/01_automation.ipynb` 를 실행하고 결과를 요약해」 (= week07 폴더에서 `python ../run_nb.py notebooks/01_automation.ipynb` · 출력이 노트북에 저장됨). 직접 보려면 JupyterLab 에서 셀을 위에서 아래로 실행한다. 코드는 저장소 스크립트를 부르기만 한다 — 바꿀 것은 `data/params.csv` 뿐.

In [ ]:
# 0) 저장소 확인 · 기본 = 로컬 week07 폴더(Claude Code · JupyterLab) → 아래 if 는 건너뜀 · (선택) Colab 이면 clone
import os, pathlib
if not pathlib.Path('scripts/render_inputs.py').exists():
    !git clone https://github.com/sentirain7/2026_MOOC_Lecture_Materials.git repo
    %cd repo/week07
print(os.getcwd())

In [ ]:
# 1) LAMMPS 가 있는지 확인 — 기본 = conda 환경 mdai 에 포함(../environment.yml) · (선택) Colab 에서만 아래 두 줄의 주석을 푼다
# !pip -q install condacolab && python -c 'import condacolab; condacolab.install()'   # Colab 1회 · 재시작됨
# !conda install -q -y -c conda-forge lammps
import shutil; LMP = shutil.which('lmp') or shutil.which('lmp_serial') or 'lmp'
print('lmp =', LMP)

In [ ]:
# 2) ① 템플릿 치환 — data/params.csv 6줄 → out/inputs/in.* 6개 (+ manifest.json 해시)
!python3 scripts/render_inputs.py
print(open('out/inputs/in.T10_s11').read()[:600])

In [ ]:
# 3) ② 반복 실행 + 실패 감지 + 재시도 1회 — 6건 (T12_s22 는 Δt 0.03 으로 일부러 실패 → 0.015 재시도)
!python3 scripts/run_batch.py --lmp {LMP}
import pandas as pd; pd.read_csv('out/runs/status.csv')

In [ ]:
# 4) ③ 로그 파싱 — thermo 블록 → CSV · 요약 JSON
!python3 scripts/parse_log.py
import json; s = json.load(open('out/thermo/summary.json'))
pd.DataFrame(s).T[['n','temp_mean','press_mean','toteng_mean']]

In [ ]:
# 5) 그림 — 6건의 온도 시계열 (목표 T 0.8 · 1.0 · 1.2 에 수렴) · 재시도 건은 점선
import matplotlib.pyplot as plt, glob
fig, ax = plt.subplots(figsize=(8,4.5))
for f in sorted(glob.glob('out/thermo/*.csv')):
    d = pd.read_csv(f); name = f.split('/')[-1][:-4]
    if len(d) < 3: continue
    ax.plot(d['step'], d['temp'], ls='--' if 'retry' in name else '-', label=name)
ax.set_xlabel('step'); ax.set_ylabel('T (LJ)'); ax.legend(fontsize=8, ncol=2); fig.tight_layout(); fig.savefig('out/thermo/temperature_6runs.png', dpi=120)
print('saved out/thermo/temperature_6runs.png')

## 확인
- `out/runs/status.csv` 7행 · `ok` 6 · `lost_atoms` 1 (T12_s22 시도 1) · 시도 2 는 `ok`
- `out/thermo/summary.json` 의 `temp_mean` 이 목표 T 의 ±2 % 안
- 바꿔 보기: `data/params.csv` 에 `T13_s11,1.3,11,0.005,1000` 한 줄 추가 → 2) 부터 다시 → 7건

못 돌아가면 `expected_output/` 의 같은 파일로 W08 로 간다.